# 製造 before/after 產生器

用 Bolts 測試原圖（模型在這個域表現好）+ 你 Drive 上的自訓練 best.pt，產出乾淨的「原圖 → 偵測」對照，存回 Drive 給老師合成上站。
登入**同一個放 best.pt 的 Google 帳號**，Run All 即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics roboflow

In [ ]:
import glob, os, shutil, cv2
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO
# 1) 取 Bolts 測試原圖（貼你的免費 Roboflow key）
rf = Roboflow(api_key=getpass('Roboflow API key: '))
ds = rf.workspace('bolts').project('bolts-final').version(1).download('yolov8')
imgs = sorted(glob.glob(ds.location + '/test/images/*'))[:3]
print('取得', len(imgs), '張測試圖')

In [ ]:
# 2) 載入自訓練模型（Drive 上的 best.pt）
m = YOLO('/content/drive/MyDrive/yolo-course/metal/best.pt')
# 3) 產 before(原圖)+after(偵測) 存回 Drive
dst = '/content/drive/MyDrive/yolo-course/metal/ba'; os.makedirs(dst, exist_ok=True)
for i, p in enumerate(imgs):
    shutil.copy(p, f'{dst}/before_{i}.jpg')
    r = m.predict(p, conf=0.25, verbose=False)[0]
    cv2.imwrite(f'{dst}/after_{i}.jpg', r.plot())
    print('影像', i, '偵測', len(r.boxes), '件')
drive.flush_and_unmount(); print('完成 → yolo-course/metal/ba/')